In [22]:
labels = {
    0: "billing",
    1: "account",
    2: "technical"
}

In [23]:
from datasets import Dataset

# 1. Expand the data
data = {
    "text": [
        # billing
        "My payment failed but money was deducted",
        "I was charged twice for my subscription",
        "Where can I download my invoice?",
        "My refund has not arrived",
        "Why did my subscription payment increase?",
        "I was billed after cancelling my plan",
        "My card was charged incorrectly",
        "I need a copy of my payment receipt",
        "Why was an extra fee added?",
        "My payment is still pending",
        "The refund amount is incorrect",
        "I want to update my billing information",
        "The invoice shows the wrong amount",
        "My subscription renewal charge failed",
        "I was charged for a service I did not use",

        # account
        "I forgot my password",
        "I cannot log into my account",
        "How can I change my email address?",
        "My account has been locked",
        "I am not receiving the login OTP",
        "How do I reset my password?",
        "I want to update my phone number",
        "My account verification is failing",
        "I cannot access my profile",
        "How do I delete my account?",
        "My login credentials are not working",
        "I need to change my username",
        "My account was disabled",
        "I am unable to verify my email",
        "How can I update my profile details?",

        # technical
        "The application crashes when I open it",
        "The page is not loading",
        "The app freezes after clicking submit",
        "I am getting a server error",
        "The application is extremely slow",
        "The upload feature is not working",
        "I am getting a 500 internal server error",
        "The dashboard is not displaying correctly",
        "The app closes automatically",
        "The website keeps timing out",
        "The screen goes blank after login",
        "The submit button does not work",
        "The application is stuck on loading",
        "The API is returning an error",
        "The app is not responding"
    ],

    "label": (
        [0] * 15 +
        [1] * 15 +
        [2] * 15
    )
}

In [24]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_texts, test_texts, train_labels, test_labels = train_test_split(
    data["text"],
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

test_dataset = Dataset.from_dict({
    "text": test_texts,
    "label": test_labels
})

In [25]:
#tokenizer
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [26]:
sample = tokenizer(
    "My payment failed but money was deducted"
)

print(sample)

{'input_ids': [101, 2026, 7909, 3478, 2021, 2769, 2001, 2139, 29510, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [27]:
print(
    tokenizer.convert_ids_to_tokens(
        sample["input_ids"]
    )
)

['[CLS]', 'my', 'payment', 'failed', 'but', 'money', 'was', 'de', '##ducted', '[SEP]']


In [28]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)
test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

print(train_dataset.column_names)

Map: 100%|██████████| 9/9 [00:00<?, ? examples/s]

['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [29]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3971.88it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
id2label = {
    0: "billing",
    1: "account",
    2: "technical"
}

label2id = {
    "billing": 0,
    "account": 1,
    "technical": 2
}

In [31]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6390.64it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy
    }

In [33]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=1,
    load_best_model_at_end=True,
    report_to="none"
)

In [34]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [35]:
trainer.train()

c:\Users\Priya Koma\Desktop\AI_ML\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.001958,1.044485,0.333333
2,0.960046,0.950161,0.777778
3,0.763441,0.833970,0.777778
4,0.626927,0.736376,0.888889
5,0.492402,0.663723,0.777778
6,0.393338,0.597735,0.888889
7,0.414922,0.558743,1.000000
8,0.382494,0.545577,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]
c:\Users\Priya Koma\Desktop\AI_ML\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.76s/it]
c:\Users\Priya Koma\Desktop\AI_ML\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]
c:\Users\Priya Koma\Desktop\AI_ML\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.89s/it]
c:\Users

TrainOutput(global_step=72, training_loss=0.6817153721219964, metrics={'train_runtime': 88.4309, 'train_samples_per_second': 3.257, 'train_steps_per_second': 0.814, 'total_flos': 4768911396864.0, 'train_loss': 0.6817153721219964, 'epoch': 8.0})

In [36]:
results = trainer.evaluate()

print(results)

c:\Users\Priya Koma\Desktop\AI_ML\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.382494,0.545577,8,1.000000


{'eval_loss': 0.5455772280693054, 'eval_accuracy': 1.0}


In [37]:
trainer.save_model("./support_ticket_model")

tokenizer.save_pretrained(
    "./support_ticket_model"
)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it]


('./support_ticket_model\\tokenizer_config.json',
 './support_ticket_model\\tokenizer.json')

In [38]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./support_ticket_model",
    tokenizer="./support_ticket_model"
)

text = "My application crashes whenever I upload a file"

result = classifier(text)

print(result)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6834.22it/s]

[{'label': 'technical', 'score': 0.7230517268180847}]


In [ ]:
test_messages = [
    "I was charged twice this month",
    "I forgot my account password",
    "The application shows a 500 error"
]

for message in test_messages:
    print(message)
    print(classifier(message, top_k=None))
    print()

I was charged twice this month
[{'label': 'billing', 'score': 0.6715863943099976}, {'label': 'account', 'score': 0.22205539047718048}, {'label': 'technical', 'score': 0.10635830461978912}]

I forgot my account password
[{'label': 'account', 'score': 0.6723326444625854}, {'label': 'billing', 'score': 0.22875601053237915}, {'label': 'technical', 'score': 0.0989113375544548}]

The application shows a 500 error
[{'label': 'technical', 'score': 0.7758371233940125}, {'label': 'account', 'score': 0.11391901969909668}, {'label': 'billing', 'score': 0.11024384945631027}]

I lost contact in my mobile
[{'label': 'account', 'score': 0.6301165223121643}, {'label': 'billing', 'score': 0.25316908955574036}, {'label': 'technical', 'score': 0.11671440303325653}]

